# Grafo acumulado de participación en proyectos UBACyT

Esta notebook lee la hoja `Datos` del Excel consolidado, elimina únicamente
duplicados exactos del mismo año, conserva un respaldo y genera dos CSV:
`nodos_ubacyt.csv` y `conexiones_ubacyt.csv`.

Para repetir el proceso: instalar `requirements-grafos.txt`, abrir esta notebook,
revisar los parámetros de la primera celda y ejecutar todas las celdas en orden.
El archivo de criterios está en `docs/criterios_grafo_ubacyt.md`.
Los años se toman de la base; no hay una lista fija de 2021, 2022 o 2023.
Cada fila declara Tipo de proyecto. Para una base histórica de nueve columnas,
ejecutar primero scripts/migrate_project_types.py (con respaldo y validación).

## Parámetros y dependencias

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from copy import copy
from datetime import datetime, timezone
from hashlib import sha256
from itertools import combinations
from pathlib import Path
from tempfile import NamedTemporaryFile
import json
import math
import os
import re
import shutil
import unicodedata

import pandas as pd
import openpyxl
from openpyxl.styles import NamedStyle
from openpyxl.utils import get_column_letter


def encontrar_raiz() -> Path:
    configurada = os.environ.get('GRAFO_UBACYT_RAIZ')
    if configurada:
        return Path(configurada).expanduser().resolve()
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if ((carpeta / 'output' / 'xlsx' / 'UBACYT-consolidado-2021-2022-2023.xlsx').is_file()
                or (carpeta / 'docs' / 'criterios_grafo_ubacyt.md').is_file()):
            return carpeta.resolve()
    raise FileNotFoundError('Abrí la notebook desde grafo_investigacion o definí GRAFO_UBACYT_RAIZ.')


RAIZ = encontrar_raiz()
RUTA_EXCEL = Path(os.environ.get('GRAFO_EXCEL', RAIZ / 'output' / 'xlsx' / 'UBACYT-consolidado-2021-2022-2023.xlsx'))
RUTA_SALIDA = RAIZ / 'output' / 'grafos' / 'ubacyt'
HOJA_DATOS = 'Datos'
BORRAR_DUPLICADOS_EN_EXCEL = True
ANIOS_INCLUIDOS = None  # None: todos; por ejemplo [2021, 2022] para limitar el grafo.
TIPOS_INCLUIDOS = ['UBACYT']

# Los multiplicadores por tipo de proyecto se aplican por separado a los nodos
# y las conexiones. Para incluir otro tipo, definir ambos y agregarlo arriba.
PESOS_TIPO_PROYECTO = {'UBACYT': {'nodos': 1, 'conexiones': 1}}
PUNTOS_ROL = {
    'ESTUDIANTE DE GRADO': 1,
    'ESTUDIANTE': 1,
    'INVESTIGADOR EN FORMACION': 2,
    'INVESTIGADOR TESISTA': 2,
    'INVESTIGADOR DE APOYO': 2,
    'INVESTIGADOR ASESOR': 3,
    'INVESTIGADOR FORMADO': 3,
    'CODIRECTOR': 5,
    'DIRECTOR': 6,
    'SIN ROL': 1,
}

# Solo se aplican equivalencias revisadas manualmente. No se unen nombres por
# similitud ni se eliminan sus tildes. Clave y valor: nombres completos.
ALIAS_PERSONAS = {}

COLUMNAS = [
    'Año del listado', 'Código proyecto', 'Convocatoria', 'Integrante',
    'Rol integrante', 'Título del proyecto', 'Unidad académica',
    'Página PDF', 'Fila en página',
    'Tipo de proyecto', 'fuente_completa',
]
CLAVE_DUPLICADOS = COLUMNAS[:7] + ['Tipo de proyecto', 'fuente_completa']  # Solo se ignoran las dos columnas de ubicación.
print('Base:', RUTA_EXCEL)
print('Tipos incluidos:', TIPOS_INCLUIDOS)
print('Versiones:', {'python': os.sys.version.split()[0], 'pandas': pd.__version__, 'openpyxl': openpyxl.__version__})

Base: C:\Users\tomas.bustos\Documents\personal\uba\grafo_investigacion\output\xlsx\UBACYT-consolidado-2021-2022-2023.xlsx
Tipos incluidos: ['UBACYT']
Versiones: {'python': '3.12.14', 'pandas': '3.0.1', 'openpyxl': '3.1.5'}


## Lectura de la base y deduplicación exacta

In [2]:
def normalizar_espacios(valor) -> str:
    if valor is None or (not isinstance(valor, str) and pd.isna(valor)):
        return ''
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(valor))).strip()


def normalizar_persona(valor) -> str:
    return normalizar_espacios(valor).upper()


def normalizar_rol(valor) -> str:
    texto = normalizar_espacios(valor).upper()
    texto = ''.join(c for c in unicodedata.normalize('NFKD', texto) if not unicodedata.combining(c))
    texto = re.sub(r'/A\b', '', texto)  # DIRECTOR/A/A también pasa a DIRECTOR.
    texto = re.sub(r'CO\s*-\s*DIRECTOR', 'CODIRECTOR', texto)
    return texto or 'SIN ROL'


def detectar_tipo(codigo, convocatoria='') -> str:
    codigo = normalizar_espacios(codigo).upper()
    convocatoria = normalizar_espacios(convocatoria).upper()
    if codigo.startswith('PDE'):
        return 'PDE'
    if codigo.startswith('PIUBA'):
        return 'PIUBAS'
    if convocatoria.startswith('UBACYT') or re.fullmatch(r'\d{14,16}(?:BA)?', codigo):
        return 'UBACYT'
    return 'NO_CLASIFICADO'


def id_persona(nombre: str) -> str:
    return 'p_' + sha256(nombre.encode('utf-8')).hexdigest()[:20]


def normalizar_tipo(valor) -> str:
    tipo = normalizar_espacios(valor).upper()
    if not tipo:
        raise ValueError('Falta Tipo de proyecto. Las bases ampliadas deben informarlo explícitamente.')
    return tipo


def leer_base(ruta: Path) -> pd.DataFrame:
    datos = pd.read_excel(ruta, sheet_name=HOJA_DATOS, dtype=object, keep_default_na=False, engine='openpyxl')
    faltantes = [c for c in COLUMNAS if c not in datos.columns]
    if faltantes:
        raise ValueError(f'Faltan columnas obligatorias: {faltantes}')
    if datos.columns.duplicated().any():
        raise ValueError('La base tiene nombres de columnas duplicados.')
    adicionales = [c for c in datos.columns if c not in COLUMNAS]
    if adicionales:
        raise ValueError(f'Hay columnas nuevas: {adicionales}. Actualizá COLUMNAS y la clave de duplicados; no se descartaron datos.')
    datos = datos.loc[:, COLUMNAS].copy()
    assert datos['fuente_completa'].isin(['completa','sólo dirección y codirección']).all(), 'Cobertura de fuente inválida'
    datos['Tipo de proyecto'].map(normalizar_tipo)  # Validar sin alterar los datos originales.
    if datos['Código proyecto'].map(normalizar_espacios).eq('').any():
        raise ValueError('Hay filas sin código de proyecto; revisarlas antes de generar el grafo.')
    if datos['Integrante'].map(normalizar_espacios).eq('').any():
        raise ValueError('Hay filas sin nombre de persona; revisarlas antes de generar el grafo.')
    for columna in ['Año del listado', 'Página PDF', 'Fila en página']:
        numeros = pd.to_numeric(datos[columna], errors='raise')
        if not numeros.map(lambda n: math.isfinite(n) and int(n) == n).all():
            raise ValueError(f'{columna} debe contener enteros finitos.')
        datos[columna] = numeros.astype('int64')
    return datos


def deduplicar_base(datos: pd.DataFrame):
    # Comparación exacta: NO cambia mayúsculas, espacios, saltos de línea ni roles.
    repetida = datos.duplicated(subset=CLAVE_DUPLICADOS, keep='first')
    eliminadas = datos.loc[repetida].copy()
    eliminadas.insert(0, 'Fila Excel original', eliminadas.index + 2)
    limpia = datos.loc[~repetida].copy().reset_index(drop=True)
    return limpia, eliminadas, (~repetida).tolist()


def actualizar_excel_sin_duplicados(ruta: Path, limpia: pd.DataFrame, conservar: list[bool]):
    if all(conservar):
        return None
    original_hash = sha256(ruta.read_bytes()).hexdigest()
    carpeta_respaldos = ruta.parent / 'respaldos'
    carpeta_respaldos.mkdir(parents=True, exist_ok=True)
    respaldo = carpeta_respaldos / f'{ruta.stem}.antes_deduplicar_{original_hash[:12]}.xlsx'
    if respaldo.exists():
        if sha256(respaldo.read_bytes()).hexdigest() != original_hash:
            raise RuntimeError('Existe un respaldo con contenido diferente. No se sobrescribió.')
    else:
        shutil.copy2(ruta, respaldo)
    libro = openpyxl.load_workbook(ruta)
    hoja = libro[HOJA_DATOS]
    alturas = [hoja.row_dimensions[i+2].height for i, queda in enumerate(conservar) if queda]
    estilos = []
    for indice in range(1, len(COLUMNAS)+1):
        celda = hoja.cell(2, indice)
        nombre = f'GrafoDatosColumna{indice}'
        if nombre not in libro.named_styles:
            estilo = NamedStyle(name=nombre, font=copy(celda.font), fill=copy(celda.fill),
                                border=copy(celda.border), alignment=copy(celda.alignment),
                                number_format=celda.number_format, protection=copy(celda.protection))
            libro.add_named_style(estilo)
        estilos.append(nombre)
    hoja.delete_rows(2, len(conservar))
    for numero, valores in enumerate(limpia.itertuples(index=False, name=None), start=2):
        hoja.append(valores)
        for indice, estilo in enumerate(estilos, start=1):
            hoja.cell(numero, indice).style = estilo
        hoja.row_dimensions[numero].height = alturas[numero-2]
    for indice in list(hoja.row_dimensions):
        if indice > len(limpia)+1:
            del hoja.row_dimensions[indice]
    nueva_referencia = f'A1:{get_column_letter(len(COLUMNAS))}{len(limpia)+1}'
    for tabla in hoja.tables.values():
        tabla.ref = nueva_referencia
        if tabla.autoFilter is not None:
            tabla.autoFilter.ref = nueva_referencia
    if hoja.auto_filter.ref:
        hoja.auto_filter.ref = nueva_referencia
    if 'Control' in libro.sheetnames:
        control = libro['Control']
        control['H3'] = 'Filas conservadas'
        for propiedad in ['font', 'fill', 'border', 'alignment', 'protection']:
            setattr(control['H3'], propiedad, copy(getattr(control['G3'], propiedad)))
        cantidades = limpia.groupby('Año del listado').size().to_dict()
        for fila in range(4, control.max_row+1):
            anio = control.cell(fila, 1).value
            if not isinstance(anio,(int,float)):
                break
            if anio in cantidades:
                control.cell(fila, 8, int(cantidades[anio])).number_format = '#,##0'
        control['A9'] = 'Datos sin duplicados. Los recuentos de extracción son los originales.'
        cantidad_roles_vacios = int(limpia['Rol integrante'].map(normalizar_espacios).eq('').sum())
        control['A10'] = f'Los {cantidad_roles_vacios} roles vacíos originales se conservaron en la base.'
        control['J9'] = 'Deduplicación de Datos'
        control['J10'] = 'Filas eliminadas'
        control['K10'] = int(len(conservar)-len(limpia))
        control['J11'] = 'Filas conservadas'
        control['K11'] = int(len(limpia))
        control['K10'].number_format = control['K11'].number_format = '#,##0'
    with NamedTemporaryFile(prefix='deduplicado_', suffix='.xlsx', dir=ruta.parent, delete=False) as handle:
        temporal = Path(handle.name)
    try:
        libro.save(temporal)
        libro.close()
        comprobacion = leer_base(temporal)
        pd.testing.assert_frame_equal(comprobacion.reset_index(drop=True), limpia.reset_index(drop=True), check_dtype=False)
        if comprobacion.duplicated(subset=CLAVE_DUPLICADOS).any():
            raise AssertionError('Quedaron duplicados exactos en el Excel temporal.')
        os.replace(temporal, ruta)
    finally:
        libro.close()
        if temporal.exists():
            temporal.unlink()
    return respaldo

## Revisar duplicados y actualizar el Excel con respaldo

In [3]:
datos_originales = leer_base(RUTA_EXCEL)
hash_entrada = sha256(RUTA_EXCEL.read_bytes()).hexdigest()
datos_limpios, filas_eliminadas, conservar = deduplicar_base(datos_originales)
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)
ejecucion = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUTA_AUDITORIA = RUTA_SALIDA / 'auditoria' / ejecucion
RUTA_AUDITORIA.mkdir(parents=True, exist_ok=False)
filas_eliminadas.to_csv(RUTA_AUDITORIA/'filas_duplicadas_eliminadas.csv',index=False,encoding='utf-8-sig',lineterminator='\n')
print('Filas leídas:', len(datos_originales))
print('Duplicados exactos eliminados:', len(filas_eliminadas))
print('Filas conservadas:', len(datos_limpios))
print(datos_limpios.groupby('Año del listado').size().rename('filas').to_string())
respaldo = None
if BORRAR_DUPLICADOS_EN_EXCEL and len(filas_eliminadas):
    respaldo = actualizar_excel_sin_duplicados(RUTA_EXCEL, datos_limpios, conservar)
    print('Excel actualizado. Respaldo:', respaldo)
(RUTA_AUDITORIA/'limpieza.json').write_text(json.dumps({
    'filas_entrada':len(datos_originales),'duplicados_eliminados':len(filas_eliminadas),
    'filas_conservadas':len(datos_limpios),'hash_entrada':hash_entrada,
    'respaldo':str(respaldo) if respaldo else None,'clave_duplicados':CLAVE_DUPLICADOS,
},ensure_ascii=False,indent=2),encoding='utf-8')

Filas leídas: 46919
Duplicados exactos eliminados: 0
Filas conservadas: 46919
Año del listado
2021    14961
2022    14536
2023    13671
2024     1900
2026     1851


426

## Identificar personas, tipos de proyecto y roles

In [4]:
def preparar_participaciones(datos: pd.DataFrame, anios=None, tipos=None,
                             puntos_rol=None, pesos_tipo=None, alias_personas=None):
    tipos = TIPOS_INCLUIDOS if tipos is None else tipos
    puntos_rol = PUNTOS_ROL if puntos_rol is None else puntos_rol
    pesos_tipo = PESOS_TIPO_PROYECTO if pesos_tipo is None else pesos_tipo
    alias_personas = ALIAS_PERSONAS if alias_personas is None else alias_personas
    for tipo in tipos:
        if tipo not in pesos_tipo or set(pesos_tipo[tipo]) != {'nodos', 'conexiones'}:
            raise ValueError(f'Faltan los multiplicadores de nodos y conexiones para {tipo}.')
        if any(not isinstance(v, (int, float)) or isinstance(v, bool) or not math.isfinite(v) or v <= 0
               for v in pesos_tipo[tipo].values()):
            raise ValueError(f'Multiplicadores inválidos para {tipo}.')
    if any(not isinstance(v, (int, float)) or isinstance(v, bool) or not math.isfinite(v) or v < 0
           for v in puntos_rol.values()):
        raise ValueError('Los puntos de rol deben ser números finitos no negativos.')
    trabajo = datos.copy()
    if anios is not None:
        if not anios:
            raise ValueError('La selección de años está vacía.')
        ausentes = set(anios)-set(trabajo['Año del listado'])
        if ausentes:
            raise ValueError(f'Años no encontrados: {sorted(ausentes)}')
        trabajo = trabajo[trabajo['Año del listado'].isin(anios)].copy()
    trabajo['tipo_proyecto'] = trabajo['Tipo de proyecto'].map(normalizar_tipo)
    no_clasificados = trabajo[trabajo['tipo_proyecto'].eq('NO_CLASIFICADO')]
    if len(no_clasificados):
        raise ValueError('Hay códigos de tipo no reconocido: '+str(sorted(no_clasificados['Código proyecto'].unique())[:15]))
    excluidos = trabajo[~trabajo['tipo_proyecto'].isin(tipos)].copy()
    trabajo = trabajo[trabajo['tipo_proyecto'].isin(tipos)].copy()
    if trabajo.empty:
        raise ValueError('No hay participaciones para los tipos y años seleccionados.')
    trabajo['codigo_proyecto'] = trabajo['Código proyecto'].map(lambda v: normalizar_espacios(v).upper())
    trabajo['persona'] = trabajo['Integrante'].map(normalizar_persona)
    alias = {normalizar_persona(k):normalizar_persona(v) for k,v in alias_personas.items()}
    if any(not k or not v for k,v in alias.items()):
        raise ValueError('Los alias deben tener nombres completos no vacíos.')
    if any(v in alias and alias[v] != v for v in alias.values()):
        raise ValueError('Usá alias directos hacia el nombre final, sin cadenas ni ciclos.')
    trabajo['persona'] = trabajo['persona'].map(lambda v: alias.get(v,v))
    trabajo['rol'] = trabajo['Rol integrante'].map(normalizar_rol)
    faltantes = sorted(set(trabajo['rol'])-set(puntos_rol))
    if faltantes:
        raise ValueError(f'Faltan puntajes para estos roles: {faltantes}. Agregalos a PUNTOS_ROL antes de continuar.')
    trabajo['puntos_rol'] = trabajo['rol'].map(puntos_rol)
    trabajo['Id'] = trabajo['persona'].map(id_persona)
    identidades = trabajo[['Id','persona']].drop_duplicates()
    if identidades['Id'].duplicated().any():
        raise ValueError('Colisión de identificadores de persona.')
    roles_detectados = (trabajo.assign(rol_original=trabajo['Rol integrante'].map(normalizar_espacios))
                       .groupby(['rol_original','rol','puntos_rol'],dropna=False).size().rename('filas').reset_index())
    clave = ['tipo_proyecto','codigo_proyecto','persona','Id']
    participaciones = (trabajo.groupby(clave,sort=True,dropna=False)
                      .agg(puntos_rol=('puntos_rol','max'),
                           roles_observados=('rol',lambda valores:'; '.join(sorted(set(valores)))),
                           anios_observados=('Año del listado',lambda valores:'; '.join(map(str,sorted(set(valores))))))
                      .reset_index())
    # Si hay empate entre roles con igual puntaje, se conserva la lista de roles;
    # la contribución al prestigio se cuenta una sola vez por proyecto.
    participaciones['peso_tipo_nodo'] = participaciones['tipo_proyecto'].map(lambda t:pesos_tipo[t]['nodos'])
    participaciones['aporte_prestigio'] = participaciones['puntos_rol']*participaciones['peso_tipo_nodo']
    return trabajo, participaciones, excluidos, roles_detectados


trabajo, participaciones, excluidos, roles_detectados = preparar_participaciones(datos_limpios, anios=ANIOS_INCLUIDOS)
print('Filas UBACyT consideradas:', len(trabajo))
print('Filas de otros tipos excluidas:', len(excluidos))
print('Participaciones persona-proyecto distintas:', len(participaciones))
print('Proyectos distintos:', participaciones[['tipo_proyecto','codigo_proyecto']].drop_duplicates().shape[0])
print(roles_detectados[['rol','puntos_rol','filas']].groupby(['rol','puntos_rol']).sum().to_string())

Filas UBACyT consideradas: 44664
Filas de otros tipos excluidas: 2255
Participaciones persona-proyecto distintas: 25595
Proyectos distintos: 3225
                                      filas
rol                       puntos_rol       
CODIRECTOR                5            2160
DIRECTOR                  6            6855
ESTUDIANTE                1              58
ESTUDIANTE DE GRADO       1            5096
INVESTIGADOR ASESOR       3            9633
INVESTIGADOR DE APOYO     2               3
INVESTIGADOR EN FORMACION 2           11480
INVESTIGADOR FORMADO      3            9359
INVESTIGADOR TESISTA      2               2
SIN ROL                   1              18


## Construir nodos y conexiones

In [5]:
def construir_grafos(participaciones: pd.DataFrame, pesos_tipo=None):
    pesos_tipo = PESOS_TIPO_PROYECTO if pesos_tipo is None else pesos_tipo
    nodos = (participaciones.groupby(['Id','persona'],sort=True,as_index=False)['aporte_prestigio'].sum()
             .rename(columns={'persona':'Label','aporte_prestigio':'prestigio'})
             .loc[:,['Id','Label','prestigio']].sort_values('Id').reset_index(drop=True))
    pesos = Counter()
    for (tipo,codigo), grupo in participaciones.groupby(['tipo_proyecto','codigo_proyecto'],sort=True):
        personas = sorted(set(grupo['Id']))
        multiplicador = pesos_tipo[tipo]['conexiones']
        for origen,destino in combinations(personas,2):
            pesos[origen,destino] += multiplicador
    conexiones = pd.DataFrame([(a,b,'Undirected',peso) for (a,b),peso in sorted(pesos.items())],
                               columns=['Source','Target','Type','peso'])
    return nodos, conexiones


nodos, conexiones = construir_grafos(participaciones)
print('Nodos:', len(nodos))
print('Conexiones no dirigidas:', len(conexiones))
print('Suma del prestigio:', nodos['prestigio'].sum())
print('Suma de los pesos de las conexiones:', conexiones['peso'].sum())
print(nodos.head(5).to_string(index=False))
print(conexiones.head(5).to_string(index=False))

Nodos: 16959
Conexiones no dirigidas: 119769
Suma del prestigio: 76560
Suma de los pesos de las conexiones: 138406
                    Id                     Label  prestigio
p_00020eaae2634f53b4c1      SABSAY, LETICIA INES          3
p_0003a671ba331ad01a0d  DUBOIS, VERONICA ANGELES          4
p_000837788fa53943cf61       AMBROS, LUIS ADRIÁN         14
p_00096ed54913f3023f19 GATTI, MARÍA LAURA AMALIA         24
p_000a98506a13607b06ae  DENTATI, LUCIANA CELESTE          6
                Source                 Target       Type  peso
p_00020eaae2634f53b4c1 p_30abd4ec882c1aee5aee Undirected     1
p_00020eaae2634f53b4c1 p_4cfb82755c29be239d98 Undirected     1
p_00020eaae2634f53b4c1 p_71f0f76b12f0ea9434be Undirected     1
p_00020eaae2634f53b4c1 p_785f52585dca6a0d3786 Undirected     1
p_00020eaae2634f53b4c1 p_bec8cf9f57d5e2fd8448 Undirected     1


## Validar toda la red y comprobar reproducibilidad

In [6]:
def validar_grafos(nodos, conexiones, participaciones, pesos_tipo=None):
    pesos_tipo = PESOS_TIPO_PROYECTO if pesos_tipo is None else pesos_tipo
    ids = set(nodos['Id'])
    assert nodos['Id'].is_unique
    assert len(nodos) == participaciones['persona'].nunique()
    assert set(participaciones['Id']) == ids
    assert not participaciones.duplicated(['tipo_proyecto','codigo_proyecto','Id']).any()
    assert not conexiones.duplicated(['Source','Target']).any()
    assert conexiones['Source'].lt(conexiones['Target']).all()
    assert set(conexiones['Source']).union(conexiones['Target']).issubset(ids)
    assert conexiones['Type'].eq('Undirected').all()
    assert conexiones['peso'].gt(0).all()
    assert nodos['prestigio'].ge(0).all()
    assert math.isclose(float(nodos['prestigio'].sum()),float(participaciones['aporte_prestigio'].sum()))
    proyectos_por_persona = defaultdict(set)
    proyectos_y_miembros = {}
    for (tipo,codigo),grupo in participaciones.groupby(['tipo_proyecto','codigo_proyecto']):
        miembros = set(grupo['Id'])
        proyectos_y_miembros[tipo,codigo] = miembros
        for identificador in miembros:
            proyectos_por_persona[identificador].add((tipo,codigo))
    total_esperado = sum(math.comb(len(miembros),2)*pesos_tipo[tipo]['conexiones']
                         for (tipo,codigo),miembros in proyectos_y_miembros.items())
    assert math.isclose(float(conexiones['peso'].sum()),float(total_esperado))
    # Verificación independiente de CADA conexión mediante intersección de grupos.
    for origen,destino,_,peso in conexiones.itertuples(index=False,name=None):
        comunes = proyectos_por_persona[origen] & proyectos_por_persona[destino]
        esperado = sum(pesos_tipo[tipo]['conexiones'] for tipo,codigo in comunes)
        assert math.isclose(float(peso),float(esperado))
    return {'nodos':len(nodos),'conexiones':len(conexiones),
            'participaciones':len(participaciones),'proyectos':len(proyectos_y_miembros),
            'conexiones_cotejadas':len(conexiones),
            'nodos_aislados':len(ids-set(conexiones['Source']).union(conexiones['Target']))}


validacion = validar_grafos(nodos, conexiones, participaciones)
assert not datos_limpios.duplicated(subset=CLAVE_DUPLICADOS).any()
otra_limpieza, otras_eliminadas, _ = deduplicar_base(datos_limpios)
pd.testing.assert_frame_equal(datos_limpios, otra_limpieza)
assert len(otras_eliminadas)==0

# Agregar un corte anual idéntico no debe aumentar prestigio ni peso de vínculos.
otro_corte = trabajo[COLUMNAS].copy()
otro_corte['Año del listado'] = otro_corte['Año del listado'] + 100
base_prueba = pd.concat([trabajo[COLUMNAS],otro_corte],ignore_index=True)
_, mismas_participaciones, _, _ = preparar_participaciones(base_prueba)
mismos_nodos, mismas_conexiones = construir_grafos(mismas_participaciones)
pd.testing.assert_frame_equal(nodos, mismos_nodos)
pd.testing.assert_frame_equal(conexiones, mismas_conexiones)
print('Validaciones correctas:', validacion)
print('Deduplicación idempotente y ausencia de inflación por repetir años: correctas.')

Validaciones correctas: {'nodos': 16959, 'conexiones': 119769, 'participaciones': 25595, 'proyectos': 3225, 'conexiones_cotejadas': 119769, 'nodos_aislados': 39}
Deduplicación idempotente y ausencia de inflación por repetir años: correctas.


## Casos pequeños con resultados conocidos

In [7]:
# Casos pequeños con resultados esperados calculados explícitamente.
def fila_prueba(anio,codigo,persona,rol,pagina=1,tipo=None):
    return [anio,codigo,'',persona,rol,'Título de prueba','Unidad de prueba',pagina,1,tipo or detectar_tipo(codigo),'completa']


ejemplo = pd.DataFrame([
    fila_prueba(2021,'20020190100001BA','PERSONA A','ESTUDIANTE DE GRADO'),
    fila_prueba(2021,'20020190100001BA','PERSONA A','ESTUDIANTE DE GRADO',2),
    fila_prueba(2022,'20020190100001BA','PERSONA A','DIRECTOR'),
    fila_prueba(2021,'20020190100001BA','PERSONA B','INVESTIGADOR FORMADO'),
    fila_prueba(2022,'20020190100001BA','PERSONA B','INVESTIGADOR FORMADO'),
    fila_prueba(2021,'20020190100002BA','PERSONA A','INVESTIGADOR FORMADO'),
    fila_prueba(2021,'20020190100002BA','PERSONA B','INVESTIGADOR DE APOYO'),
    fila_prueba(2021,'20020190100003BA','PERSONA C',''),
    fila_prueba(2021,'PDE_1_2021','PERSONA D','DIRECTOR'),
    fila_prueba(2021,'PDE_1_2021','PERSONA E','ESTUDIANTE DE GRADO'),
],columns=COLUMNAS)
ejemplo_limpio, ejemplo_duplicados, _ = deduplicar_base(ejemplo)
assert len(ejemplo_duplicados)==1 and len(ejemplo_limpio)==9
puntos_prueba={'ESTUDIANTE DE GRADO':1,'DIRECTOR':6,'INVESTIGADOR FORMADO':3,'INVESTIGADOR DE APOYO':2,'SIN ROL':1}
pesos_prueba={'UBACYT':{'nodos':1,'conexiones':1}}
_, miembros_prueba, _, _ = preparar_participaciones(ejemplo_limpio,tipos=['UBACYT'],
                                                  puntos_rol=puntos_prueba,pesos_tipo=pesos_prueba,alias_personas={})
n_prueba,e_prueba=construir_grafos(miembros_prueba,pesos_prueba)
assert n_prueba.set_index('Label')['prestigio'].to_dict()=={'PERSONA A':9,'PERSONA B':5,'PERSONA C':1}
assert len(e_prueba)==1 and e_prueba.iloc[0]['peso']==2
assert validar_grafos(n_prueba,e_prueba,miembros_prueba,pesos_prueba)['nodos_aislados']==1
rol_desconocido=ejemplo_limpio.copy()
rol_desconocido.loc[0,'Rol integrante']='ROL NO DEFINIDO'
try:
    preparar_participaciones(rol_desconocido,tipos=['UBACYT'],puntos_rol=puntos_prueba,pesos_tipo=pesos_prueba,alias_personas={})
except ValueError as error:
    assert 'ROL NO DEFINIDO' in str(error)
else:
    raise AssertionError('Un rol nuevo no puede recibir un puntaje silenciosamente.')
pesos_ampliados={'UBACYT':{'nodos':1,'conexiones':1},'PDE':{'nodos':3,'conexiones':2}}
_, m_ampliados, _, _=preparar_participaciones(ejemplo_limpio,tipos=['UBACYT','PDE'],
                                            puntos_rol=puntos_prueba,pesos_tipo=pesos_ampliados,alias_personas={})
n_ampliados,e_ampliados=construir_grafos(m_ampliados,pesos_ampliados)
assert n_ampliados.set_index('Label').loc['PERSONA D','prestigio']==18
assert n_ampliados.set_index('Label').loc['PERSONA E','prestigio']==3
assert e_ampliados.loc[(e_ampliados['Source']==min(id_persona('PERSONA D'),id_persona('PERSONA E')))
                      & (e_ampliados['Target']==max(id_persona('PERSONA D'),id_persona('PERSONA E'))),'peso'].iloc[0]==2
colision = pd.DataFrame([
    fila_prueba(2021,'CODIGO-COMPARTIDO','PERSONA A','DIRECTOR',tipo='UBACYT'),
    fila_prueba(2021,'CODIGO-COMPARTIDO','PERSONA B','ESTUDIANTE DE GRADO',tipo='UBACYT'),
    fila_prueba(2021,'CODIGO-COMPARTIDO','PERSONA A','DIRECTOR',tipo='PICT'),
    fila_prueba(2021,'CODIGO-COMPARTIDO','PERSONA B','ESTUDIANTE DE GRADO',tipo='PICT'),
],columns=COLUMNAS)
colision_limpia, copias, _ = deduplicar_base(colision)
assert len(copias)==0 and len(colision_limpia)==4
pesos_colision={'UBACYT':{'nodos':1,'conexiones':1},'PICT':{'nodos':2,'conexiones':3}}
_, pc, _, _=preparar_participaciones(colision_limpia,tipos=['UBACYT','PICT'],puntos_rol=puntos_prueba,pesos_tipo=pesos_colision,alias_personas={})
nc,ec=construir_grafos(pc,pesos_colision)
assert nc.set_index('Label')['prestigio'].to_dict()=={'PERSONA A':18,'PERSONA B':3}
assert len(ec)==1 and ec.iloc[0]['peso']==4
try:
    preparar_participaciones(colision_limpia,tipos=['UBACYT','PICT'],puntos_rol=puntos_prueba,pesos_tipo=pesos_prueba,alias_personas={})
except ValueError as error:
    assert 'PICT' in str(error)
else:
    raise AssertionError('Un tipo incluido sin ponderación debe detener el proceso.')
print('Pruebas correctas: duplicados, roles, aislados, pesos por tipo y códigos iguales de financiamientos distintos.')

Pruebas correctas: duplicados, roles, aislados, pesos por tipo y códigos iguales de financiamientos distintos.


## Exportar los dos CSV y la auditoría

In [8]:
def exportar_csv_atomico(datos: pd.DataFrame, ruta: Path):
    with NamedTemporaryFile(prefix='export_', suffix='.csv', dir=ruta.parent, delete=False) as handle:
        temporal = Path(handle.name)
    try:
        datos.to_csv(temporal,index=False,encoding='utf-8-sig',lineterminator='\n')
        os.replace(temporal,ruta)
    finally:
        if temporal.exists():temporal.unlink()


RUTA_NODOS = RUTA_SALIDA / 'nodos_ubacyt.csv'
RUTA_CONEXIONES = RUTA_SALIDA / 'conexiones_ubacyt.csv'
exportar_csv_atomico(nodos,RUTA_NODOS)
exportar_csv_atomico(conexiones,RUTA_CONEXIONES)
exportar_csv_atomico(filas_eliminadas,RUTA_AUDITORIA/'filas_duplicadas_eliminadas.csv')
exportar_csv_atomico(roles_detectados,RUTA_AUDITORIA/'roles_detectados.csv')
exportar_csv_atomico(participaciones,RUTA_AUDITORIA/'participaciones_persona_proyecto.csv')
exportar_csv_atomico(excluidos,RUTA_AUDITORIA/'filas_otros_programas_excluidas.csv')

resumen = {
    'fecha_utc':ejecucion,'archivo_base':str(RUTA_EXCEL),'hoja':HOJA_DATOS,
    'hash_base_entrada':hash_entrada,'hash_base_salida':sha256(RUTA_EXCEL.read_bytes()).hexdigest(),
    'respaldo':str(respaldo) if respaldo else None,
    'filas_entrada':len(datos_originales),'duplicados_eliminados':len(filas_eliminadas),
    'filas_base_limpia':len(datos_limpios),'filas_grafo':len(trabajo),'filas_otros_programas':len(excluidos),
    'anios':sorted(map(int,trabajo['Año del listado'].unique())),
    'tipos_incluidos':TIPOS_INCLUIDOS,'puntos_rol':PUNTOS_ROL,'pesos_tipo':PESOS_TIPO_PROYECTO,
    'columnas_fuente':COLUMNAS,'clave_duplicados':CLAVE_DUPLICADOS,
    'tipos_presentes':datos_limpios['Tipo de proyecto'].map(normalizar_tipo).value_counts().to_dict(),
    'criterio_rol_persona_proyecto':'máximo puntaje observado',
    'identidad_persona':'NFC + mayúsculas + espacios normalizados, conservando tildes y puntuación; alias explícitos',
    'validacion':validacion,
    'suma_prestigio':float(nodos['prestigio'].sum()),'suma_pesos_conexiones':float(conexiones['peso'].sum()),
    'sha256_nodos':sha256(RUTA_NODOS.read_bytes()).hexdigest(),
    'sha256_conexiones':sha256(RUTA_CONEXIONES.read_bytes()).hexdigest(),
    'versiones':{'python':os.sys.version.split()[0],'pandas':pd.__version__,'openpyxl':openpyxl.__version__},
}
(RUTA_AUDITORIA/'resumen_ejecucion.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
(RUTA_SALIDA/'ultima_ejecucion.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')

# Relectura de ambos CSV, con los nombres que usarán los importadores de grafos.
pd.testing.assert_frame_equal(pd.read_csv(RUTA_NODOS,encoding='utf-8-sig',keep_default_na=False),nodos,check_dtype=False)
pd.testing.assert_frame_equal(pd.read_csv(RUTA_CONEXIONES,encoding='utf-8-sig',keep_default_na=False),conexiones,check_dtype=False)
if BORRAR_DUPLICADOS_EN_EXCEL:
    pd.testing.assert_frame_equal(leer_base(RUTA_EXCEL),datos_limpios,check_dtype=False)
print('Nodos:',RUTA_NODOS)
print('Conexiones:',RUTA_CONEXIONES)
print('Auditoría:',RUTA_AUDITORIA)
print('Archivos guardados y releídos sin diferencias.')

Nodos: C:\Users\tomas.bustos\Documents\personal\uba\grafo_investigacion\output\grafos\ubacyt\nodos_ubacyt.csv
Conexiones: C:\Users\tomas.bustos\Documents\personal\uba\grafo_investigacion\output\grafos\ubacyt\conexiones_ubacyt.csv
Auditoría: C:\Users\tomas.bustos\Documents\personal\uba\grafo_investigacion\output\grafos\ubacyt\auditoria\20261110T134500621299Z
Archivos guardados y releídos sin diferencias.
